# AI 5102 - Exercise 1: Language Model APIs

In this hands-on exercise you will learn to work with Large Language Models programmatically using APIs.

**You will learn about:**
* Setting up API access for NVIDIA and Anthropic
* Steering LLM behavior with system messages, temperature, and token limits
* Understanding tokens and how they affect API costs
* Building multi-turn conversations that maintain context
* Designing and implementing your own chatbot

**Connection to Course Concepts:** This exercise accompanies Session 2, where we explore how to interact with Large Language Models programmatically. Understanding these fundamentals is essential for all advanced topics we'll cover, including tool use, agents, and RAG.

---

## Before You Begin

Please make a copy of this Python notebook into your Google Drive or onto your own computer.  If you edit it directly without making a copy, your changes will be lost.

**IMPORTANT:**
To assist our grading efforts, we have code and markdown cells with special annotations like these:
```
# === STUDENT INPUT CELL: Exercise X ===
<!-- === STUDENT INPUT CELL: Exercise Y === -->
```
To avoid grading issues, we ask you to not remove or alter these. So please take extra precaution when you select-all and paste. Finally, only make changes in the cells with annotations. If you create additional cells during development, consolidate the final solution code and make sure your answer is in the notebook cell we are expecting.

### Expected Cost
This exercise should cost **less than \$1 total** if you follow the instructions. Most exercises use `meta/llama-3.1-8b-instruct` and `claude-haiku-4-5-20251001`, which are the most cost-effective models available (roughly 1/4 the cost of their larger siblings).

### Setting Up Spending Limits (Required)

Before starting, set up your API keys and spending limits:

**NVIDIA:**
1. Go to https://build.nvidia.com/settings/api-keys
2. Generate your API key for free

**Anthropic:**
1. Go to https://console.anthropic.com/settings/limits
2. Set monthly spend limit to \$10

If you exceed \$5 on this assignment, contact the instructor.

### Prerequisites
This exercise assumes familiarity with Python basics from the pre-work, including:
- Dictionaries and lists
- JSON-like data structures
- Basic string manipulation

---

## Point Distribution

| Section | Points |
|---------|--------|
| Part 1: NVIDIA API Basics | 15 |
| Part 2: Temperature & Tokens | 10 |
| Part 3: Anthropic Comparison | 10 |
| Part 4: Understanding Tokens | 15 |
| Part 5: Multi-turn Conversations | 15 |
| Part 6: Build Your Own Chatbot | 35 |
| **Total** | **100** |

---

# Part 1: NVIDIA API Basics (15 points)

API Reference: https://build.nvidia.com/settings/api-keys

**Getting your API keys:**

1. Go to https://build.nvidia.com/ and create an account/log on
2. Access https://build.nvidia.com/settings/api-keys
3. Generate your API key

In [ ]:
%%capture
!pip install openai
!pip install anthropic
!pip install tiktoken

In [ ]:
from getpass import getpass
from openai import OpenAI
import os

print('Enter NVIDIA API key:')
nvidia_api_key = getpass()
nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=nvidia_api_key
)

NVIDIA_MODEL = "meta/llama-3.1-8b-instruct"

## Basic API Call

The core of the OpenAI API is the `chat.completions.create()` method. You send a list of messages, and the model returns a response.

In [ ]:
# Basic Example
messages = [
    {"role": "user", "content": "Hello, how are you?"}
]
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=messages
)
completion = response.choices[0].message.content
print(completion)

## System Messages

A **system message** lets you set the behavior, personality, and constraints for the assistant. This is one of the most powerful ways to customize LLM responses.

**Why does this work?** During RLHF (Reinforcement Learning from Human Feedback — which we will learn about later in the course), models were trained to follow instructions in the system message. Human raters preferred responses that adhered to the given persona and constraints, so the model learned to treat system messages as authoritative instructions.

In [ ]:
# Example of using a system message
messages = [
    {"role": "system", "content": """You are a pirate. You redirect all conversation
to tempt users to join your crew as part of an adventure."""},
    {"role": "user", "content": "What's the capital of France?"}
]
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=messages
)
completion = response.choices[0].message.content
print(completion)

---

## Exercise 1.1: Shakespearean System Prompt (5 points)

Create a system prompt that makes the assistant respond as a Shakespearean actor. Test it with 3 different user inputs and show the outputs.

## Parsing & Validating Responses

So far we've assumed `response.choices[0].message.content` always contains useful text. In real applications, you must **validate** the response before using it: the content could be empty (e.g., if the model was cut off or refused), or the response object could be malformed.

Below is a small helper, `extract_content()`, that safely pulls the text out of an OpenAI chat completion response and returns a clear error dictionary if anything looks wrong, instead of letting your program crash with an `IndexError` or `AttributeError`.

In [ ]:
# === Safe response parsing/validation ===

def extract_content(response):
    """Safely extract text content from an OpenAI chat completion response.

    Returns a dict:
      {"ok": True, "content": <str>}                      on success
      {"ok": False, "error": <str>}                       on failure

    This never raises — it's meant to be called on every API response
    before you use the text, so a malformed or empty response fails
    loudly and clearly instead of crashing deep inside your program.
    """
    try:
        choices = getattr(response, "choices", None)
        if not choices:
            return {"ok": False, "error": "Response has no choices"}

        message = getattr(choices[0], "message", None)
        if message is None:
            return {"ok": False, "error": "First choice has no message"}

        content = getattr(message, "content", None)
        if content is None or content.strip() == "":
            return {"ok": False, "error": "Message content is empty"}

        return {"ok": True, "content": content}
    except Exception as e:
        return {"ok": False, "error": f"Unexpected error while parsing response: {e}"}


# Demo: a normal response
demo_response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=[{"role": "user", "content": "Say hello in one word."}]
)
result = extract_content(demo_response)
print("Valid response ->", result)

## Error & Rate-Limit Handling

Production applications call LLM APIs many times, and calls occasionally fail for **transient** reasons: the server is briefly overloaded, you've hit a rate limit (`openai.RateLimitError`), or there's a network hiccup. These errors usually go away if you simply retry — ideally with **exponential backoff** (waiting longer between each retry) so you don't hammer the API while it's struggling.

Below, `call_with_retry()` wraps any NVIDIA API call. It:
1. Tries the call.
2. If it hits a rate-limit or a transient server error, waits `base_delay * 2**attempt` seconds (with a little random jitter) and tries again.
3. Gives up and re-raises the last error after `retries` attempts.

This is the same overall pattern used by production SDKs internally (many API client libraries retry 429 and 5xx errors automatically) — here we implement it ourselves so you understand what's happening.

In [ ]:
# === Robust retry wrapper with exponential backoff ===
import time
import random
import openai


def call_with_retry(client, retries=3, base_delay=1.0, **kwargs):
    """Call client.chat.completions.create(**kwargs), retrying on transient errors.

    Args:
        client: an OpenAI client instance.
        retries: maximum number of attempts (including the first try).
        base_delay: base number of seconds to wait before the first retry;
                    doubles on each subsequent retry (exponential backoff).
        **kwargs: forwarded to client.chat.completions.create(), e.g.
                  model=..., messages=..., max_tokens=...

    Returns:
        The API response object on success.

    Raises:
        The last exception encountered, if all attempts fail. Non-transient
        errors (e.g. bad request / invalid API key) are NOT retried — they
        are raised immediately, since retrying won't help.
    """
    last_exception = None

    for attempt in range(retries):
        try:
            return client.chat.completions.create(**kwargs)
        except openai.RateLimitError as e:
            last_exception = e
        except openai.APIConnectionError as e:
            last_exception = e
        except openai.APIStatusError as e:
            # Retry server-side errors (5xx); don't retry client errors (4xx
            # other than 429, which is RateLimitError and already handled above).
            if getattr(e, "status_code", 500) >= 500:
                last_exception = e
            else:
                raise

        if attempt < retries - 1:
            delay = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
            print(f"  Attempt {attempt + 1}/{retries} failed ({last_exception}); "
                  f"retrying in {delay:.1f}s...")
            time.sleep(delay)

    raise last_exception


# Demo: a normal call still works exactly the same way
demo_response = call_with_retry(
    nvidia_client,
    retries=3,
    base_delay=1.0,
    model=NVIDIA_MODEL,
    messages=[{"role": "user", "content": "Say hello in one word."}]
)
print(extract_content(demo_response))

In [ ]:
# === STUDENT INPUT CELL: Exercise 1.1 ===

# TODO: Define your Shakespearean system prompt
shakespearean_system_prompt = """
YOUR SYSTEM PROMPT HERE
"""

# Test 1
messages = [
    {"role": "system", "content": shakespearean_system_prompt},
    {"role": "user", "content": "YOUR FIRST TEST INPUT"}
]
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=messages
)
print("Test 1:")
print(response.choices[0].message.content)
print("\n---\n")

# Test 2
messages = [
    {"role": "system", "content": shakespearean_system_prompt},
    {"role": "user", "content": "YOUR SECOND TEST INPUT"}
]
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=messages
)
print("Test 2:")
print(response.choices[0].message.content)
print("\n---\n")

# Test 3
messages = [
    {"role": "system", "content": shakespearean_system_prompt},
    {"role": "user", "content": "YOUR THIRD TEST INPUT"}
]
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=messages
)
print("Test 3:")
print(response.choices[0].message.content)

---

## Exercise 1.2: Custom Use Case System Prompt (5 points)

Create a system prompt for a specific use case of your choice. Examples:
- A patient tutor for a specific subject
- A customer service representative for a fictional company
- A game character (quest giver, shopkeeper, etc.)
- A coding assistant with a specific specialty

Describe your use case and show a sample 2-turn conversation.

<!-- === STUDENT INPUT CELL: Exercise 1.2.1 === -->

**My use case:** (Describe your chosen use case in 1-2 sentences)

*YOUR DESCRIPTION HERE*

In [ ]:
# === STUDENT INPUT CELL: Exercise 1.2.2 ===

# TODO: Define your custom system prompt
custom_system_prompt = """
YOUR SYSTEM PROMPT HERE
"""

# First turn
messages = [
    {"role": "system", "content": custom_system_prompt},
    {"role": "user", "content": "YOUR FIRST MESSAGE"}
]
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=messages
)
assistant_response_1 = response.choices[0].message.content
print("User: YOUR FIRST MESSAGE")
print(f"Assistant: {assistant_response_1}")
print("\n---\n")

# Second turn (continuing the conversation)
messages.append({"role": "assistant", "content": assistant_response_1})
messages.append({"role": "user", "content": "YOUR SECOND MESSAGE"})
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=messages
)
print("User: YOUR SECOND MESSAGE")
print(f"Assistant: {response.choices[0].message.content}")

---

# Part 2: Temperature and Token Limits (10 points)

Two important parameters control LLM output: **temperature** (randomness) and **max_tokens** (length limit).

## Token Limits

LLM outputs are measured in **tokens**, not words. The `max_tokens` parameter limits how long the response can be.

In [ ]:
# Example: max_tokens cuts off the response
messages = [
    {"role": "user", "content": "Hello, how are you?"}
]
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=messages,
    max_tokens=5
)
print("With max_tokens=5:")
print(response.choices[0].message.content)

print("\n---\n")

# With a higher limit, we get the full response
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=messages,
    max_tokens=100
)
print("With max_tokens=100:")
print(response.choices[0].message.content)

In [ ]:
# What happens if max_tokens is too large?
# Each model has a maximum context window. Let's see the error:
try:
    response = nvidia_client.chat.completions.create(
        model=NVIDIA_MODEL,
        messages=[{"role": "user", "content": "Hello"}],
        max_tokens=100000
    )
except Exception as e:
    print(f"Error: {e}")
    print("\n--> The hosted NVIDIA model/API will reject values that exceed its "
          "supported generation/context limits.")


## Temperature: Controlling Randomness

### Conceptual Understanding

When an LLM generates text, it predicts a **probability distribution** over all possible next tokens. For example, after "The cat sat on the", the model might assign:
- "mat" → 40%
- "floor" → 25%
- "couch" → 15%
- "roof" → 8%
- ... (thousands of other tokens with smaller probabilities)

**Temperature** controls how the model samples from this distribution:

- **Temperature = 0**: Always pick the highest probability token ("mat"). Outputs are deterministic and repetitive.
- **Temperature = 1**: Sample according to the original probabilities. Good balance of coherence and variety.
- **Temperature > 1**: Flatten the distribution, making unlikely tokens more probable. At temperature=2, outputs become chaotic as the model samples nearly randomly.

Mathematically, temperature divides the log-probabilities before applying softmax: `P_new = softmax(logits / temperature)`. Lower temperature makes the distribution "sharper" (more peaked), higher temperature makes it "flatter" (more uniform).

In [ ]:
# Example showing effect of temperature
def generate_poem_with_temperature(temperature):
    messages = [
        {"role": "user", "content": "Give me a haiku about swans"}
    ]
    response = nvidia_client.chat.completions.create(
        model=NVIDIA_MODEL,
        messages=messages,
        max_tokens=100,
        temperature=temperature
    )
    print(response.choices[0].message.content)
    print("---")

print("===== Temperature 0 (deterministic) =====")
generate_poem_with_temperature(0)
generate_poem_with_temperature(0)
generate_poem_with_temperature(0)

print("\n===== Temperature 1 (balanced) =====")
generate_poem_with_temperature(1)
generate_poem_with_temperature(1)
generate_poem_with_temperature(1)

print("\n===== Temperature 2 (chaotic) =====")
generate_poem_with_temperature(2)
generate_poem_with_temperature(2)

### Seeing the Token Probabilities Directly

The above explanation mentions probability distributions, but we can actually **see** them using the `logprobs` parameter. This lets you peek inside the model's decision-making process.

Setting `logprobs=True` and `top_logprobs=5` returns the top 5 candidate tokens at each position, along with their log-probabilities. Combined with `max_tokens=1`, we can see exactly what the model was considering for the next token.

In [ ]:
# Illustrating the next-token distribution for "The cat sat on the "
# Using logprobs=True, top_logprobs=5, and max_tokens=1 to see the top 5 candidate tokens
import math

prompt = "The cat sat on the "
response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=[{"role": "user", "content": f"Complete this sentence with one word: {prompt}"}],
    max_tokens=1,
    logprobs=True,
    top_logprobs=5,
    temperature=0.0
)

content = response.choices[0].logprobs.content
chosen = response.choices[0].message.content

print(f'Prompt: "{prompt}"')
print(f"Chosen next token: {chosen!r}")
print("\nTop 5 candidate next tokens:")
print("-" * 50)
for i, top in enumerate(content[0].top_logprobs, start=1):
    p = math.exp(top.logprob) * 100
    print(f"  {i}. {top.token!r:12}  logprob={top.logprob:8.4f}  →  {p:5.2f}%")

## top_p (Nucleus Sampling)

`temperature` reshapes the *whole* probability distribution (making it sharper or flatter), while `top_p` restricts sampling to a smaller pool of tokens in the first place: with `top_p=0.1`, the model only samples from the smallest set of tokens whose cumulative probability reaches 10%. In other words, temperature changes *how randomly* you sample; top_p changes *how many* candidate tokens are even eligible to be sampled. They can be combined, but it's easiest to reason about them by varying one at a time.

The example below fixes `temperature=1` and varies `top_p` to show how a narrower nucleus produces safer, more predictable completions, while a wider nucleus (close to 1.0) allows more diverse — sometimes surprising — word choices.

In [ ]:
# === Varying top_p (nucleus sampling) ===

def generate_with_top_p(top_p, n=3):
    """Generate n completions at a fixed temperature, varying only top_p,
    to show how the size of the sampling pool affects output diversity."""
    for _ in range(n):
        response = nvidia_client.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[{"role": "user", "content": "Give me a haiku about swans"}],
            max_tokens=100,
            temperature=1,
            top_p=top_p
        )
        print(response.choices[0].message.content)
        print("---")


print("===== top_p = 0.1 (narrow nucleus, low diversity) =====")
generate_with_top_p(0.1)

print("\n===== top_p = 1.0 (full distribution, high diversity) =====")
generate_with_top_p(1.0)

---

## Exercise 2.1: Choosing Temperature (5 points)

For each scenario below, choose an appropriate temperature value and justify your choice in 1-2 sentences.

In [ ]:
# === STUDENT INPUT CELL: Exercise 2.1 ===

# (a) Code generation - writing a function to sort a list
temperature_a = None  # TODO: Choose a value (0, 0.3, 0.7, or 1)
justification_a = """
YOUR JUSTIFICATION HERE
"""

# (b) Creative story writing - a fantasy adventure opening
temperature_b = None  # TODO: Choose a value
justification_b = """
YOUR JUSTIFICATION HERE
"""

# (c) Factual Q&A - answering "What is the capital of France?"
temperature_c = None  # TODO: Choose a value
justification_c = """
YOUR JUSTIFICATION HERE
"""

print(f"(a) Code generation: temperature = {temperature_a}")
print(justification_a)
print(f"(b) Creative writing: temperature = {temperature_b}")
print(justification_b)
print(f"(c) Factual Q&A: temperature = {temperature_c}")
print(justification_c)

---

# Part 3: Anthropic API (10 points)

Now let's work with Anthropic's Claude models. The API is very similar to OpenAI's.

API Reference: https://docs.anthropic.com/en/api/getting-started

**Getting your API keys:**

1. Go to https://console.anthropic.com/dashboard and create an account/log on
2. Add a payment method to your account
3. Go to https://console.anthropic.com/settings/keys

In [ ]:
import anthropic

print('Enter Anthropic API key:')
anthropic_api_key = getpass()
anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)

In [ ]:
# Basic Anthropic usage
# Note: Anthropic uses "messages" without a separate system message in the list
# The system message is passed as a separate parameter

response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Give me a haiku about swans"}
    ]
)
print(response.content[0].text)

---

## Exercise 3.1: Comparing Llama and Claude (10 points)

Send the SAME prompt to both `meta/llama-3.1-8b-instruct` and `claude-haiku-4-5-20251001`. Compare the responses.

Choose a prompt that's interesting to compare (e.g., a creative task, an explanation, or a problem-solving request).

In [ ]:
# === STUDENT INPUT CELL: Exercise 3.1.1 ===

# TODO: Choose a prompt to compare
comparison_prompt = "YOUR PROMPT HERE"

# NVIDIA response
nvidia_response = nvidia_client.chat.completions.create(
    model=NVIDIA_MODEL,
    messages=[{"role": "user", "content": comparison_prompt}],
    max_tokens=200
)
print("===== NVIDIA Llama 3.1 8B =====")
print(nvidia_response.choices[0].message.content)

print("\n" + "="*50 + "\n")

# Claude Haiku response
claude_response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    messages=[{"role": "user", "content": comparison_prompt}]
)
print("===== Claude Haiku =====")
print(claude_response.content[0].text)

<!-- === STUDENT INPUT CELL: Exercise 3.1.2 === -->
**What differences do you notice in style, length, or content?**

*YOUR COMPARISON HERE (3-4 sentences)*

---

# Part 4: Understanding Tokens (15 points)

### What are Tokens?

LLMs don't process text character-by-character or word-by-word. Instead, they use **tokens**—chunks of text that can be words, parts of words, or even individual characters.

Modern LLMs use **Byte Pair Encoding (BPE)** to create their vocabulary:
1. Start with individual characters as tokens
2. Repeatedly merge the most common adjacent pairs
3. End up with a vocabulary of ~50,000-100,000 tokens

Common words like "the" are single tokens, while rare words get split into pieces. For example, "tokenization" might become ["token", "ization"].

**Why tokens matter:**
- **Context limits**: Models have maximum token limits (e.g., 128K for GPT-4.1)
- **Cost**: APIs charge per token (input and output separately)
- **Speed**: More tokens = longer generation time

**Important:** Even when two models use the same tokenization algorithm (BPE), their **token IDs can differ** because each model is trained on different data with different vocabularies. The string "hello" might be token ID 15339 in one model and token ID 9906 in another. This is why a tokenizer is always tied to a specific model. `tiktoken` only ships OpenAI tokenizers, so for the Llama models we call through NVIDIA's API we use `tiktoken.get_encoding("cl100k_base")` as an **approximation** — close enough for cost estimates, but not the model's true tokenizer. (The exact Llama tokenizer is available via Hugging Face `transformers` if you need precise counts.)

You can explore tokenization interactively at: https://platform.openai.com/tokenizer

In [ ]:
import tiktoken

def get_token_breakdown(text, model=NVIDIA_MODEL):
    """Show how text is split into tokens."""
    encoding = tiktoken.get_encoding("cl100k_base")  # approximation: tiktoken has no Llama tokenizer
    tokens = encoding.encode(text)
    token_texts = [encoding.decode([token]) for token in tokens]
    return token_texts

# Examples
print("'Hello there.' ->", get_token_breakdown("Hello there."))
print("'hello   there.' ->", get_token_breakdown("hello   there."))  # Extra spaces = more tokens
print("'antidisestablishmentarianism' ->", get_token_breakdown("antidisestablishmentarianism"))

In [ ]:
# Token counting functions
def count_tokens_nvidia(text, model="meta/llama-3.1-8b-instruct"):
    encoding = tiktoken.get_encoding("cl100k_base")  # approximation: tiktoken has no Llama tokenizer
    return len(encoding.encode(text))

def count_tokens_anthropic(text, model="claude-haiku-4-5-20251001"):
    response = anthropic_client.messages.count_tokens(
        model=model,
        messages=[{"role": "user", "content": text}]
    )
    return response.input_tokens

# Compare token counts
text = "Grape flavored Gatorade"
print(f"Text: '{text}'")
print(f"NVIDIA (approx., cl100k_base): {count_tokens_nvidia(text)} tokens")
print(f"Anthropic (claude-haiku): {count_tokens_anthropic(text)} tokens")

---

## Exercise 4.1: Tokenizing Different Text Types (5 points)

Tokenize 3 different types of text and compare the token-to-word ratio:
- (a) English prose (a sentence from a book or article)
- (b) Python code (a simple function)
- (c) Text in a non-English language (if you know one) or technical jargon

In [ ]:
# === STUDENT INPUT CELL: Exercise 4.1.1 ===

# (a) English prose
text_a = "YOUR ENGLISH PROSE HERE"

# (b) Python code
text_b = """def hello():
    print('Hello, world!')
"""

# (c) Non-English or technical text
text_c = "YOUR TEXT HERE"

for label, text in [("English prose", text_a), ("Python code", text_b), ("Other", text_c)]:
    tokens = get_token_breakdown(text)
    words = text.split()
    print(f"\n{label}:")
    print(f"  Text: {text[:50]}..." if len(text) > 50 else f"  Text: {text}")
    print(f"  Words: {len(words)}, Tokens: {len(tokens)}")
    print(f"  Tokens per word: {len(tokens)/len(words):.2f}")
    print(f"  Token breakdown: {tokens[:10]}..." if len(tokens) > 10 else f"  Token breakdown: {tokens}")

<!-- === STUDENT INPUT CELL: Exercise 4.1.2 === -->

**Which uses the most tokens per word? Why might this be?**

*YOUR ANSWER HERE (2-3 sentences)*

---

## Exercise 4.2: Cost Calculation (5 points)

Calculate the cost of a hypothetical conversation using `gpt-4.1-nano`.

**Scenario:** A chatbot has a 10-turn conversation with average 150 input tokens and 200 output tokens per turn.

**Pricing (as of 2025, check https://openai.com/pricing for current rates):**
- gpt-4.1-nano: \$0.10 per 1M input tokens, \$0.40 per 1M output tokens

Note: Prices change over time. The specific numbers matter less than understanding the calculation.

In [ ]:
# === STUDENT INPUT CELL: Exercise 4.2 ===

# TODO: Calculate the cost

# Given values
turns = 10
avg_input_tokens_per_turn = 150
avg_output_tokens_per_turn = 200

# Pricing per 1M tokens for your NVIDIA endpoint.
input_price_per_million = 0.0
output_price_per_million = 0.0

# Your calculation here
total_input_tokens = None  # TODO
total_output_tokens = None  # TODO
input_cost = None  # TODO
output_cost = None  # TODO
total_cost = None  # TODO

print(f"Total input tokens: {total_input_tokens}")
print(f"Total output tokens: {total_output_tokens}")
print(f"Input cost: ${input_cost:.6f}")
print(f"Output cost: ${output_cost:.6f}")
print(f"Total cost: ${total_cost:.6f}")

---

# Part 5: Multi-turn Conversations (15 points)

LLM APIs are **stateless**—each API call is independent, with no memory of previous calls. To create a conversation, you must send the **entire conversation history** with each request.

This is a crucial concept: the model doesn't "remember" anything. You're responsible for maintaining context by including all previous messages.

In [ ]:
# Simple conversation loop - the core pattern

messages = [{"role": "system", "content": "You are a helpful assistant."}]

print("Chat started. Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    # Add user message to history
    messages.append({"role": "user", "content": user_input})

    # Send entire history to API
    response = nvidia_client.chat.completions.create(
        model=NVIDIA_MODEL,
        messages=messages
    )

    # Get and display response
    reply = response.choices[0].message.content
    print(f"Assistant: {reply}\n")

    # Add assistant response to history
    messages.append({"role": "assistant", "content": reply})

print("\nConversation ended.")

### Why This Works

Look at the `messages` list. Each API call sends the **complete conversation history**:
1. System message (always first)
2. User message 1
3. Assistant response 1
4. User message 2
5. Assistant response 2
6. ... and so on

The model uses its attention mechanism to process this entire context and generate a coherent response. This is also why longer conversations cost more—you're sending more tokens with each turn!

---

## Exercise 5.1: Conversation with Turn Counter (10 points)

Modify the conversation loop to:
1. Include a turn counter that displays the current turn number
2. Stop automatically after 5 turns
3. After stopping, print the final messages list
4. Count the total tokens in the final messages list

In [ ]:
# === STUDENT INPUT CELL: Exercise 5.1.1 ===

# TODO: Implement the modified conversation loop

messages = [{"role": "system", "content": "You are a helpful assistant."}]
turn = 0
max_turns = 5
total_tokens = 0


print(f"Chat started. Will end after {max_turns} turns.\n")

# YOUR CODE HERE
# Hint: Modify the while loop to use turn counter instead of checking for 'exit'
# YOUR CODE HERE
# Hint: Modify the while loop to use turn counter instead of checking for 'exit'


# After the loop ends:
print("\n" + "="*50)
print("Final messages list:")
for i, msg in enumerate(messages):
    print(f"{i}: [{msg['role']}] {msg['content'][:50]}..." if len(msg['content']) > 50 else f"{i}: [{msg['role']}] {msg['content']}")

# Count total tokens
total_tokens = sum(count_tokens_nvidia(msg['content']) for msg in messages)
print(f"\nTotal tokens in conversation: {total_tokens}")

---

## Exercise 5.2: Cost and Latency Analysis (5 points)

Answer the following question:

**What happens to cost and latency as conversations get longer? Why?**

<!-- === STUDENT INPUT CELL: Exercise 5.2 === -->

*YOUR ANSWER HERE (3-4 sentences explaining the relationship between conversation length, cost, and latency)*

---

# Part 6: Build Your Own Chatbot (35 points)

Design and implement a chatbot for a specific purpose.

## Requirements

1. **Choose a use case** - Examples:
   - Quiz bot that tests knowledge on a topic
   - Writing tutor that helps improve essays
   - Recipe helper that suggests meals based on ingredients
   - Language practice partner for learning a new language
   - Interview prep coach that asks practice questions
   - Game character (NPC shopkeeper, quest giver, etc.)
   - Technical support bot for a fictional product

2. **Write a detailed system prompt** (at least 100 words) that defines:
   - The persona and personality
   - The bot's goals and constraints
   - How it should handle different types of inputs

3. **Implement using the simple conversation loop pattern**

4. **Have at least a 5-turn conversation** demonstrating your bot

5. **Write a 1-paragraph reflection** on what worked well and what limitations you noticed

## Grading Rubric

| Component | Points |
|-----------|--------|
| System prompt quality (detailed, specific, effective) | 15 |
| Working implementation | 10 |
| Demonstration conversation (5+ turns, shows capabilities) | 5 |
| Reflection (thoughtful analysis) | 5 |

<!-- === STUDENT INPUT CELL: Exercise 6.1 === -->
## My Chatbot: [YOUR CHATBOT NAME]

**Use case:** (Describe your chosen use case in 2-3 sentences)

*YOUR DESCRIPTION HERE*

In [ ]:
# === STUDENT INPUT CELL: Exercise 6.2 ===

# Your system prompt (at least 100 words)
chatbot_system_prompt = """
YOUR DETAILED SYSTEM PROMPT HERE

Include:
- Persona and personality
- Goals and constraints
- How to handle different inputs

(Delete these instructions and write your own prompt)
"""

# Word count check
word_count = len(chatbot_system_prompt.split())
print(f"System prompt word count: {word_count}")
if word_count < 100:
    print("Warning: System prompt should be at least 100 words!")

In [ ]:
# Your chatbot implementation
# Use the simple conversation loop pattern from Part 5

messages = [{"role": "system", "content": chatbot_system_prompt}]

print("Chatbot started. Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    messages.append({"role": "user", "content": user_input})

    response = nvidia_client.chat.completions.create(
        model=NVIDIA_MODEL,
        messages=messages
    )

    reply = response.choices[0].message.content
    print(f"Bot: {reply}\n")

    messages.append({"role": "assistant", "content": reply})

print("\nChatbot session ended.")

<!-- === STUDENT INPUT CELL: Exercise 6.3 === -->

## Demonstration Conversation

After running the chatbot above, copy and paste your conversation here (at least 5 turns):

```
You: [your message 1]
Bot: [bot response 1]

You: [your message 2]
Bot: [bot response 2]

... (continue for 5+ turns)
```

<!-- === STUDENT INPUT CELL: Exercise 6.4 === -->

## Reflection

Write a 1-paragraph reflection (4-6 sentences) addressing:
- What worked well with your chatbot?
- What limitations did you notice?
- If you had more time, what would you improve?

*YOUR REFLECTION HERE*

---

# Optional: Advanced Topics (0 points, enrichment)

This section covers more advanced patterns you might want to use in larger projects.

## Wrapper Classes for API Calls

For production applications, you often want to:
- Track costs across multiple calls
- Handle errors gracefully
- Log conversation history
- Support multiple models

Here's an example of a wrapper class that adds cost tracking:

In [ ]:
class NVIDIA_LLM_Caller:
    """A wrapper class for NVIDIA's OpenAI-compatible API.

    Use this when you need to:
    - Track total costs across many API calls
    - Keep a history of prompts and completions
    - Easily switch between NVIDIA-hosted models
    """

    # Pricing per 1M tokens. Set these to the rates for your endpoint.
    # The default below assumes a free endpoint.
    PRICING = {
        "meta/llama-3.1-8b-instruct": {"input": 0.0, "output": 0.0},
    }

    def __init__(self, model, api_key, input_price=None, output_price=None):
        self.model = model
        self.client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=api_key
        )
        self.history = []
        self.total_cost = 0

        if input_price is not None or output_price is not None:
            self.PRICING[self.model] = {
                "input": input_price or 0.0,
                "output": output_price or 0.0
            }

    def calculate_cost(self, input_tokens, output_tokens):
        pricing = self.PRICING.get(
            self.model,
            {"input": 0.0, "output": 0.0}
        )
        input_cost = input_tokens * pricing["input"] / 1_000_000
        output_cost = output_tokens * pricing["output"] / 1_000_000
        return input_cost + output_cost

    def generate(self, messages, temperature=0.7, max_tokens=1000):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens
        )

        completion = response.choices[0].message.content or ""

        # Prefer exact usage reported by the NVIDIA API.
        usage = getattr(response, "usage", None)
        input_tokens = getattr(usage, "prompt_tokens", 0) if usage else 0
        output_tokens = getattr(usage, "completion_tokens", 0) if usage else 0

        cost = self.calculate_cost(input_tokens, output_tokens)

        self.history.append({
            "messages": messages,
            "completion": completion,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost": cost
        })
        self.total_cost += cost

        return completion

    def get_total_cost(self):
        return f"${self.total_cost:.6f}"


In [ ]:
# Example usage of the wrapper class
caller = NVIDIA_LLM_Caller(
    model=NVIDIA_MODEL,
    api_key=nvidia_api_key
)

response1 = caller.generate([{"role": "user", "content": "What is 2+2?"}])
print(f"Response 1: {response1}")

response2 = caller.generate([{"role": "user", "content": "Tell me a joke."}])
print(f"Response 2: {response2}")

print(f"\nTotal cost so far: {caller.get_total_cost()}")
print(f"Number of calls made: {len(caller.history)}")


## API Portability: Writing Provider-Agnostic Code

One challenge you may have noticed: OpenAI and Anthropic have slightly different APIs. If you want to switch providers or support multiple models, you need to rewrite code.

Several frameworks solve this problem by providing a **unified interface** across providers:

- **[LiteLLM](https://github.com/BerriAI/litellm)** - Lightweight wrapper that translates OpenAI-format calls to 100+ providers
- **[LangChain](https://langchain.com/)** - Full framework for building LLM applications with provider abstraction
- **[Kani](https://github.com/zhudotexe/kani)** - Lightweight library focused on multi-turn conversations

**Example with LiteLLM:**
```python
from litellm import completion

# Same code works for any provider!
response = completion(
    model="gpt-4.1-nano",  # or "claude-haiku-4-5-20251001"
    messages=[{"role": "user", "content": "Hello!"}]
)
```

**Extra Credit Idea:** Pick one of these frameworks and translate your Part 6 chatbot to use it. Compare the experience to using the raw APIs.

---

# Feedback Questions

In [ ]:
# === STUDENT INPUT CELL: Feedback ===

# How many hours did you spend on this assignment? Just an approximation is fine.
num_hours_spent = 0

# What did you enjoy most about this assignment?
feedback_a = """

"""

# What part(s) of this assignment do you think should be improved?
feedback_b = """

"""